# Modul 14: Neuronale Netze mit NumPy und CNN-Bausteine | Lösungen

## Überblick

Sie implementieren Neuronen, Dense-Schichten, Aktivierungen, Softmax und Kreuzentropie mit NumPy. Darauf aufbauend prüfen Sie Gradienten, trainieren ein kleines Zwei-Schichten-MLP und untersuchen Mini-Batches, Momentum, Regularisierung sowie grundlegende Faltungs- und Poolingoperationen.

**Zugehörige Vorlesungen**

- **Vorwärts und Rückwärts**
- **MLP und Faltungen**

## Lernziele

Nach der Bearbeitung können Sie:

- Vorwärtspropagation durch Dense-Schichten mit passenden Aktivierungen und stabiler Softmax berechnen.
- Backpropagation mit der Kettenregel implementieren und Gradienten numerisch kontrollieren.
- ein kleines MLP trainieren und die Tensorformen von Faltung, Padding und Pooling planen.

## Geprüfte Fähigkeiten

- Matrixformen, ReLU, Sigmoid, Softmax und Kreuzentropie
- analytische und numerische Gradienten, Mini-Batches, Momentum und L2-Regularisierung
- 2D-Faltung, Padding, MaxPooling und Ausgabeformeln

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle erzeugt einen kleinen zweidimensionalen Moon-Datensatz, teilt ihn in Training, Validierung und Test und standardisiert ausschließlich anhand des Trainingsanteils. Alle Netzberechnungen verwenden nur NumPy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

X_gesamt, y_gesamt = make_moons(n_samples=600, noise=0.18, random_state=RANDOM_SEED)
X_train_roh, X_test_roh, y_train, y_test = train_test_split(
    X_gesamt,
    y_gesamt,
    test_size=0.20,
    stratify=y_gesamt,
    random_state=RANDOM_SEED,
)
X_train_roh, X_val_roh, y_train, y_val = train_test_split(
    X_train_roh,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=RANDOM_SEED,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_roh)
X_val = scaler.transform(X_val_roh)
X_test = scaler.transform(X_test_roh)

print("Train, Validierung, Test:", X_train.shape, X_val.shape, X_test.shape)

### Aufgabe 1: Neuron, Dense-Schicht und Aktivierungen implementieren

Implementieren Sie Funktionen für:

1. ein einzelnes künstliches Neuron `x @ w + b`,
2. eine Dense-Schicht `X @ W + b`,
3. ReLU und Sigmoid.

Testen Sie die Funktionen mit dem vorgegebenen Mini-Batch. Prüfen Sie ausdrücklich die Formen und erklären Sie, warum der Bias-Vektor über alle Batchzeilen gebroadcastet werden kann.

In [ ]:
mini_batch = np.array([[1.0, -2.0], [0.5, 1.5], [-1.0, 0.25]])
gewichte_neuron = np.array([0.8, -0.4])
bias_neuron = 0.2

gewichte_schicht = np.array([[0.6, -0.2, 0.4], [-0.5, 0.9, 0.1]])
bias_schicht = np.array([0.1, -0.3, 0.2])

def neuron_vorwaerts(x, w, b):
    pass

def dense_vorwaerts(X, W, b):
    pass

def relu(z):
    pass

def sigmoid(z):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def neuron_vorwaerts(x, w, b):
    """Berechnet die gewichtete Summe eines einzelnen Beispiels."""
    return np.asarray(x) @ np.asarray(w) + b


def dense_vorwaerts(X, W, b):
    """Berechnet alle Neuronen einer Dense-Schicht für einen vollständigen Batch."""
    return np.asarray(X) @ np.asarray(W) + np.asarray(b)


def relu(z):
    """Setzt negative Voraktivierungen auf null."""
    return np.maximum(0.0, z)


def sigmoid(z):
    """Berechnet die Sigmoidfunktion mit begrenzten Eingaben für numerische Stabilität."""
    z = np.clip(z, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-z))

neuron_ausgabe = neuron_vorwaerts(mini_batch[0], gewichte_neuron, bias_neuron)
schicht_roh = dense_vorwaerts(mini_batch, gewichte_schicht, bias_schicht)
schicht_relu = relu(schicht_roh)
schicht_sigmoid = sigmoid(schicht_roh)

print("Einzelnes Neuron:", neuron_ausgabe)
print("Dense-Rohform:", schicht_roh.shape)
print("ReLU-Ausgabe:\n", np.round(schicht_relu, 3))
print("Sigmoid-Ausgabe:\n", np.round(schicht_sigmoid, 3))
assert schicht_roh.shape == (3, 3)

> **Musterantwort und Interpretation**
>
> NumPy vergleicht Broadcasting-Formen von rechts. Der Bias besitzt einen Wert pro Ausgabeneuron und passt deshalb zur letzten Achse der Batchmatrix. Derselbe Bias-Vektor wird logisch auf jede der drei Batchzeilen angewendet, ohne dass er manuell kopiert werden muss.

### Aufgabe 2: Stabile Softmax und Kreuzentropie berechnen

Implementieren Sie eine numerisch stabile Softmaxfunktion für einen Batch von Logits sowie eine mittlere Mehrklassen-Kreuzentropie für ganzzahlige Klassenlabels.

Testen Sie die Funktionen mit den vorgegebenen großen Logits. Prüfen Sie, ob jede Wahrscheinlichkeitszeile ungefähr Summe 1 besitzt. Vergleichen Sie den Verlust guter und absichtlich vertauschter Logits.

In [ ]:
logits_gut = np.array(
    [[1002.0, 998.0, 997.0], [991.0, 996.0, 990.0], [1000.0, 999.0, 1005.0]]
)
klassen = np.array([0, 1, 2])

def stabile_softmax(logits):
    pass

def kreuzentropie_mehrklassen(wahrscheinlichkeiten, y):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def stabile_softmax(logits):
    """Wandelt Logits zeilenweise in Wahrscheinlichkeiten um."""
    logits = np.asarray(logits, dtype=float)
    # Das Abziehen des Zeilenmaximums ändert die relativen Abstände nicht,
    # verhindert aber einen Überlauf von exp() bei großen positiven Zahlen.
    verschoben = logits - logits.max(axis=1, keepdims=True)
    exponenten = np.exp(verschoben)
    return exponenten / exponenten.sum(axis=1, keepdims=True)


def kreuzentropie_mehrklassen(wahrscheinlichkeiten, y):
    """Berechnet den mittleren negativen Logarithmus der richtigen Klassenwahrscheinlichkeit."""
    wahrscheinlichkeiten = np.asarray(wahrscheinlichkeiten, dtype=float)
    y = np.asarray(y, dtype=int)
    richtige_wahrscheinlichkeit = wahrscheinlichkeiten[np.arange(len(y)), y]
    return -np.mean(np.log(np.clip(richtige_wahrscheinlichkeit, 1e-12, 1.0)))

wahrscheinlichkeiten_gut = stabile_softmax(logits_gut)
verlust_gut = kreuzentropie_mehrklassen(wahrscheinlichkeiten_gut, klassen)

# Durch Spaltenumkehr erhalten die richtigen Klassen meist kleinere Wahrscheinlichkeiten.
logits_schlecht = logits_gut[:, ::-1]
wahrscheinlichkeiten_schlecht = stabile_softmax(logits_schlecht)
verlust_schlecht = kreuzentropie_mehrklassen(wahrscheinlichkeiten_schlecht, klassen)

print("Wahrscheinlichkeiten:\n", np.round(wahrscheinlichkeiten_gut, 4))
print("Zeilensummen:", wahrscheinlichkeiten_gut.sum(axis=1))
print("Verlust gute Logits:", round(verlust_gut, 4))
print("Verlust vertauschte Logits:", round(verlust_schlecht, 4))
assert np.allclose(wahrscheinlichkeiten_gut.sum(axis=1), 1.0)

> **Musterantwort und Interpretation**
>
> Für die wahre Klasse wird der negative Logarithmus ihrer vorhergesagten Wahrscheinlichkeit berechnet. Nähert sich diese Wahrscheinlichkeit null, wächst der negative Logarithmus stark. Damit erzeugt eine falsche Vorhersage mit sehr hoher Sicherheit einen deutlich größeren Lernimpuls als eine unsichere falsche Vorhersage.

### Aufgabe 3: Gradienten einer Dense-Ausgabe numerisch prüfen

Betrachten Sie ein lineares Modell `y_hat = X @ w + b` mit mittlerem quadratischem Fehler. Leiten und implementieren Sie die analytischen Gradienten nach `w` und `b`.

Berechnen Sie zusätzlich zentrale numerische Differenzen mit `epsilon = 1e-5`. Geben Sie die maximale absolute Abweichung zwischen analytischen und numerischen Gradienten aus und prüfen Sie, ob sie kleiner als `1e-6` ist.

In [ ]:
X_grad = np.array([[1.0, 2.0], [-1.0, 0.5], [0.25, -0.75], [2.0, -1.0]])
y_grad = np.array([2.0, -0.5, 0.25, 1.5])
w_grad = np.array([0.3, -0.2])
b_grad = 0.1

def mse_verlust(X, y, w, b):
    pass

def analytische_gradienten(X, y, w, b):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def mse_verlust(X, y, w, b):
    prognose = X @ w + b
    return np.mean((prognose - y) ** 2)


def analytische_gradienten(X, y, w, b):
    """Nutzt d/dz (z-y)^2 = 2(z-y) und die Kettenregel."""
    prognose = X @ w + b
    fehler = prognose - y
    d_w = (2.0 / len(y)) * X.T @ fehler
    d_b = 2.0 * fehler.mean()
    return d_w, d_b

analytisch_w, analytisch_b = analytische_gradienten(X_grad, y_grad, w_grad, b_grad)

epsilon = 1e-5
numerisch_w = np.zeros_like(w_grad)
for j in range(len(w_grad)):
    w_plus = w_grad.copy()
    w_minus = w_grad.copy()
    w_plus[j] += epsilon
    w_minus[j] -= epsilon
    numerisch_w[j] = (
        mse_verlust(X_grad, y_grad, w_plus, b_grad)
        - mse_verlust(X_grad, y_grad, w_minus, b_grad)
    ) / (2.0 * epsilon)

numerisch_b = (
    mse_verlust(X_grad, y_grad, w_grad, b_grad + epsilon)
    - mse_verlust(X_grad, y_grad, w_grad, b_grad - epsilon)
) / (2.0 * epsilon)

maximale_abweichung = max(
    np.max(np.abs(analytisch_w - numerisch_w)),
    abs(analytisch_b - numerisch_b),
)
print("Analytischer Gewichtsgradient:", analytisch_w)
print("Numerischer Gewichtsgradient:", numerisch_w)
print("Analytischer Biasgradient:", analytisch_b)
print("Numerischer Biasgradient:", numerisch_b)
print("Maximale Abweichung:", maximale_abweichung)
assert maximale_abweichung < 1e-6

> **Musterantwort und Interpretation**
>
> Er kann Vorzeichenfehler, fehlende Faktoren, falsche Matrixachsen oder eine fehlerhafte Kettenregel sichtbar machen. Numerische Differenzen sind jedoch langsam und selbst nur näherungsweise, weil epsilon einen Kompromiss zwischen Rundungs- und Approximationsfehler darstellt. Sie eignen sich zur Prüfung kleiner Beispiele, nicht als Trainingsverfahren.

### Aufgabe 4: Ein Zwei-Schichten-MLP mit Mini-Batches trainieren

Implementieren Sie ein binäres MLP mit Architektur `2 -> 12 -> 1`, ReLU in der verborgenen Schicht und Sigmoid am Ausgang. Verwenden Sie binäre Kreuzentropie, Mini-Batches der Größe 32 und einfachen stochastischen Gradientenabstieg.

Trainieren Sie höchstens 300 Epochen. Speichern Sie Trainings- und Validierungsverlust und berichten Sie Accuracy für Training, Validierung und Test. Zeichnen Sie die Verlustkurven.

In [ ]:
def initialisiere_mlp(eingaben, verborgen, seed=42):
    pass

def mlp_vorwaerts(X, parameter):
    pass

def binaere_kreuzentropie(y, p):
    pass

def mlp_gradienten(X, y, parameter, cache, l2=0.0):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def initialisiere_mlp(eingaben, verborgen, seed=42):
    lokaler_rng = np.random.default_rng(seed)
    # He-ähnliche Initialisierung passt gut zu ReLU und verhindert zu große Startwerte.
    return {
        "W1": lokaler_rng.normal(0.0, np.sqrt(2.0 / eingaben), size=(eingaben, verborgen)),
        "b1": np.zeros(verborgen),
        "W2": lokaler_rng.normal(0.0, np.sqrt(2.0 / verborgen), size=(verborgen, 1)),
        "b2": np.zeros(1),
    }


def mlp_vorwaerts(X, parameter):
    z1 = X @ parameter["W1"] + parameter["b1"]
    a1 = np.maximum(0.0, z1)
    z2 = a1 @ parameter["W2"] + parameter["b2"]
    p = sigmoid(z2).ravel()
    return p, {"X": X, "z1": z1, "a1": a1, "p": p}


def binaere_kreuzentropie(y, p):
    p = np.clip(p, 1e-8, 1.0 - 1e-8)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def mlp_gradienten(X, y, parameter, cache, l2=0.0):
    batch_groesse = len(y)
    # Für Sigmoid plus binäre Kreuzentropie vereinfacht sich dL/dz2 zu p-y.
    d_z2 = (cache["p"] - y).reshape(-1, 1) / batch_groesse
    d_W2 = cache["a1"].T @ d_z2 + l2 * parameter["W2"]
    d_b2 = d_z2.sum(axis=0)

    d_a1 = d_z2 @ parameter["W2"].T
    d_z1 = d_a1 * (cache["z1"] > 0.0)
    d_W1 = X.T @ d_z1 + l2 * parameter["W1"]
    d_b1 = d_z1.sum(axis=0)
    return {"W1": d_W1, "b1": d_b1, "W2": d_W2, "b2": d_b2}


def trainiere_mlp(
    X_train_local,
    y_train_local,
    X_val_local,
    y_val_local,
    epochen=300,
    batch_groesse=32,
    lernrate=0.05,
    l2=0.0,
    momentum=0.0,
    geduld=None,
):
    parameter = initialisiere_mlp(X_train_local.shape[1], verborgen=12, seed=RANDOM_SEED)
    geschwindigkeit = {name: np.zeros_like(wert) for name, wert in parameter.items()}
    historie = {"train_loss": [], "val_loss": []}
    bestes = {name: wert.copy() for name, wert in parameter.items()}
    bester_val_verlust = np.inf
    epochen_ohne_verbesserung = 0
    lokaler_rng = np.random.default_rng(RANDOM_SEED)

    for epoche in range(epochen):
        reihenfolge = lokaler_rng.permutation(len(X_train_local))
        for start in range(0, len(reihenfolge), batch_groesse):
            batch_index = reihenfolge[start:start + batch_groesse]
            X_batch = X_train_local[batch_index]
            y_batch = y_train_local[batch_index]

            _, cache = mlp_vorwaerts(X_batch, parameter)
            gradienten = mlp_gradienten(X_batch, y_batch, parameter, cache, l2=l2)

            for name in parameter:
                geschwindigkeit[name] = momentum * geschwindigkeit[name] - lernrate * gradienten[name]
                parameter[name] += geschwindigkeit[name]

        train_p, _ = mlp_vorwaerts(X_train_local, parameter)
        val_p, _ = mlp_vorwaerts(X_val_local, parameter)
        train_loss = binaere_kreuzentropie(y_train_local, train_p)
        val_loss = binaere_kreuzentropie(y_val_local, val_p)
        historie["train_loss"].append(train_loss)
        historie["val_loss"].append(val_loss)

        if val_loss < bester_val_verlust - 1e-5:
            bester_val_verlust = val_loss
            bestes = {name: wert.copy() for name, wert in parameter.items()}
            epochen_ohne_verbesserung = 0
        else:
            epochen_ohne_verbesserung += 1

        if geduld is not None and epochen_ohne_verbesserung >= geduld:
            break

    return bestes, historie

parameter_mlp, historie_mlp = trainiere_mlp(
    X_train,
    y_train,
    X_val,
    y_val,
    epochen=300,
    batch_groesse=32,
    lernrate=0.05,
)

for name, X_teil, y_teil in [
    ("Training", X_train, y_train),
    ("Validierung", X_val, y_val),
    ("Test", X_test, y_test),
]:
    p_teil, _ = mlp_vorwaerts(X_teil, parameter_mlp)
    klasse_teil = (p_teil >= 0.5).astype(int)
    print(name, "Accuracy:", round(accuracy_score(y_teil, klasse_teil), 3))

plt.plot(historie_mlp["train_loss"], label="Training")
plt.plot(historie_mlp["val_loss"], label="Validierung")
plt.xlabel("Epoche")
plt.ylabel("Binäre Kreuzentropie")
plt.title("MLP-Trainingsverlauf")
plt.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Trainingsdaten bestimmen die Gewichte. Validierungsdaten unterstützen Entscheidungen wie Trainingsdauer, Architektur oder Regularisierung, ohne Gradienten für die Gewichte zu liefern. Der Testdatensatz bleibt bis zum Abschluss unberührt und schätzt die Leistung nach allen Entscheidungen. Wiederholte Testauswertung würde ihn schrittweise in einen Validierungsdatensatz verwandeln.

### Aufgabe 5: Momentum, L2-Regularisierung und Early Stopping vergleichen

Trainieren Sie drei Varianten mit derselben Initialisierung:

1. einfacher SGD ohne Momentum,
2. SGD mit Momentum `0.9`,
3. Momentum `0.9`, L2-Regularisierung und Early Stopping mit Geduld 25.

Vergleichen Sie Anzahl ausgeführter Epochen, besten Validierungsverlust, Test-Accuracy und Summe der quadrierten Gewichte. Interpretieren Sie, weshalb die komplexere Trainingsstrategie nicht in jedem Lauf automatisch die höchste Testkennzahl garantieren muss.

In [ ]:
varianten = {
    "SGD": {"momentum": 0.0, "l2": 0.0, "geduld": None},
    "Momentum": {"momentum": 0.9, "l2": 0.0, "geduld": None},
    "Momentum + L2 + EarlyStopping": {"momentum": 0.9, "l2": 0.001, "geduld": 25},
}

# ============================================================
# MUSTERLÖSUNG
# ============================================================

trainingsberichte = []
varianten_parameter = {}
for name, einstellungen in varianten.items():
    aktuelle_parameter, aktuelle_historie = trainiere_mlp(
        X_train,
        y_train,
        X_val,
        y_val,
        epochen=350,
        batch_groesse=32,
        lernrate=0.035,
        l2=einstellungen["l2"],
        momentum=einstellungen["momentum"],
        geduld=einstellungen["geduld"],
    )
    varianten_parameter[name] = aktuelle_parameter
    test_p, _ = mlp_vorwaerts(X_test, aktuelle_parameter)
    test_klasse = (test_p >= 0.5).astype(int)
    gewichtsnorm_quadrat = sum(
        np.sum(wert**2)
        for parameter_name, wert in aktuelle_parameter.items()
        if parameter_name.startswith("W")
    )
    trainingsberichte.append(
        {
            "Variante": name,
            "Epochen": len(aktuelle_historie["val_loss"]),
            "Bester_Val_Verlust": min(aktuelle_historie["val_loss"]),
            "Test_Accuracy": accuracy_score(y_test, test_klasse),
            "Gewichtsnorm_Quadrat": gewichtsnorm_quadrat,
        }
    )

display(pd.DataFrame(trainingsberichte).round(4))

> **Musterantwort und Interpretation**
>
> Die Kennzahl hängt von Stichprobenzufall, Datensatzgröße, Initialisierung und Optimierungsrauschen ab. Regularisierung soll vor allem Verallgemeinerung stabilisieren und Modellkomplexität begrenzen, kann auf einem einzelnen Testsplit aber geringfügig schlechter erscheinen. Belastbare Aussagen benötigen wiederholte Splits oder Kreuzvalidierung, Streuungsangaben und einen fairen Vergleich mit identischen Daten und Startbedingungen.

### Aufgabe 6: Integrationsaufgabe: Faltung, Padding, Pooling und Tensorformen

Implementieren Sie eine 2D-Kreuzkorrelation für ein einzelnes Graustufenbild und einen Filter mit optionalem Zero-Padding und Stride. Implementieren Sie außerdem 2-mal-2-MaxPooling mit Stride 2.

Wenden Sie den vorgegebenen Kantenfilter auf das 6-mal-6-Testbild einmal mit `padding=0` und einmal mit `padding=1` an. Poolen Sie die gepaddete Ausgabe. Berechnen Sie die erwarteten räumlichen Formen zusätzlich mit der Formel

`floor((Eingabe + 2*Padding - Kernel) / Stride) + 1`

und prüfen Sie sie mit Assertions.

In [ ]:
testbild = np.array(
    [
        [0, 0, 0, 0, 0, 0],
        [0, 1, 1, 1, 0, 0],
        [0, 1, 3, 3, 1, 0],
        [0, 1, 3, 3, 1, 0],
        [0, 0, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0],
    ],
    dtype=float,
)
kantenfilter = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=float)

def ausgabelaenge(eingabe, kernel, stride=1, padding=0):
    pass

def kreuzkorrelation_2d(bild, kernel, stride=1, padding=0):
    pass

def max_pool_2d(karte, pool=2, stride=2):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def ausgabelaenge(eingabe, kernel, stride=1, padding=0):
    return int(np.floor((eingabe + 2 * padding - kernel) / stride) + 1)


def kreuzkorrelation_2d(bild, kernel, stride=1, padding=0):
    """Berechnet die in CNN-Bibliotheken übliche Kreuzkorrelation ohne Kernelspiegelung."""
    bild = np.asarray(bild, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    gepolstert = np.pad(bild, pad_width=padding, mode="constant")

    ausgabe_hoehe = ausgabelaenge(bild.shape[0], kernel.shape[0], stride, padding)
    ausgabe_breite = ausgabelaenge(bild.shape[1], kernel.shape[1], stride, padding)
    ausgabe = np.empty((ausgabe_hoehe, ausgabe_breite), dtype=float)

    for i in range(ausgabe_hoehe):
        for j in range(ausgabe_breite):
            zeile = i * stride
            spalte = j * stride
            ausschnitt = gepolstert[
                zeile:zeile + kernel.shape[0],
                spalte:spalte + kernel.shape[1],
            ]
            ausgabe[i, j] = np.sum(ausschnitt * kernel)
    return ausgabe


def max_pool_2d(karte, pool=2, stride=2):
    ausgabe_hoehe = ausgabelaenge(karte.shape[0], pool, stride, padding=0)
    ausgabe_breite = ausgabelaenge(karte.shape[1], pool, stride, padding=0)
    ausgabe = np.empty((ausgabe_hoehe, ausgabe_breite), dtype=float)
    for i in range(ausgabe_hoehe):
        for j in range(ausgabe_breite):
            ausschnitt = karte[
                i * stride:i * stride + pool,
                j * stride:j * stride + pool,
            ]
            ausgabe[i, j] = ausschnitt.max()
    return ausgabe

karte_valid = kreuzkorrelation_2d(testbild, kantenfilter, padding=0)
karte_same = kreuzkorrelation_2d(testbild, kantenfilter, padding=1)
gepoolt = max_pool_2d(karte_same, pool=2, stride=2)

print("Valid-Form:", karte_valid.shape)
print("Same-ähnliche Form:", karte_same.shape)
print("Pooling-Form:", gepoolt.shape)
print("Gepoolte Merkmalskarte:\n", gepoolt)

assert karte_valid.shape == (
    ausgabelaenge(6, 3, 1, 0),
    ausgabelaenge(6, 3, 1, 0),
)
assert karte_same.shape == (
    ausgabelaenge(6, 3, 1, 1),
    ausgabelaenge(6, 3, 1, 1),
)
assert gepoolt.shape == (
    ausgabelaenge(karte_same.shape[0], 2, 2, 0),
    ausgabelaenge(karte_same.shape[1], 2, 2, 0),
)

> **Musterantwort und Interpretation**
>
> Padding kann Randinformationen länger erhalten und die räumliche Größe einer Merkmalskarte steuern. Pooling verdichtet lokale Bereiche, reduziert Breite und Höhe sowie damit spätere Rechenkosten, verliert aber genaue Positions- und Intensitätsdetails. Architekturentscheidungen müssen deshalb den gewünschten Kompromiss zwischen Detail, Robustheit und Ressourcen berücksichtigen.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?